In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [2]:
import torch
import torch.nn as nn
from collections import deque
import os

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

relu = nn.ReLU()

def relu_derivative(x):
    return (x > 0).float()

cuda


In [4]:
def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    pos = torch.arange(0, max_seq_len).unsqueeze(1)
    i = torch.arange(0, d_model, 2)
    div = 10000 ** (i / d_model)
    PE[:, 0::2] = torch.sin(pos / div)
    PE[:, 1::2] = torch.cos(pos / div)
    return PE

In [5]:
class Attention_Block(nn.Module):
  def __init__(self, d_model:int, num_heads:int):
    super().__init__()
    self.num_heads = num_heads
    self.input = None
    self.Exp = None
    self.d_k = d_model // num_heads
    self.d_model = d_model
    self.Q_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.K_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.V_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.W_O = nn.Parameter(torch.randn(d_model, d_model))
    self.Q = None
    self.K = None
    self.V = None
    self.scores = None
    self.A = None
    self.deltaE_heads = None
    self.deltaE_cat = None

  def forward_pass(self, E, mask=False):
    n = E.shape[0]
    self.inputs = E
    self.Exp = self.inputs.unsqueeze(0).expand(self.num_heads, -1, -1)
    self.Q = torch.bmm(self.Exp, self.Q_M)
    self.K = torch.bmm(self.Exp, self.K_M)
    self.V = torch.bmm(self.Exp, self.V_M)

    self.scores = torch.bmm(self.Q, self.K.transpose(-2, -1)) / (self.d_k) ** 0.5
    if mask:
      m = torch.triu(torch.ones(n, n), diagonal=1).bool().to(self.Q_M.device)
      self.scores = self.scores.masked_fill(m, float('-inf')).to(device)

    self.A = torch.softmax(self.scores, dim=-1)
    self.deltaE_heads = torch.bmm(self.A, self.V)
    self.deltaE_cat = self.deltaE_heads.transpose(0, 1).reshape(n, -1)

    return self.deltaE_cat @ self.W_O

  def backpropagation(self, i_g):
    n = self.inputs.shape[0]

    dW_O = self.deltaE_cat.T @ i_g
    d_deltaE_cat = i_g @ self.W_O.T

    # split gradient back to each head
    d_deltaE_heads = d_deltaE_cat.reshape(n, self.num_heads, self.d_k).transpose(0, 1)

    dA = torch.bmm(d_deltaE_heads, self.V.transpose(-2, -1))
    dV = torch.bmm(self.A.transpose(-2, -1), d_deltaE_heads)

    # softmax jacobian
    dS = self.A * (dA - (dA * self.A).sum(dim=-1, keepdim=True))

    dQ = torch.bmm(dS, self.K) / self.d_k ** 0.5
    dK = torch.bmm(dS.transpose(-2, -1), self.Q) / self.d_k ** 0.5

    dW_Q = torch.bmm(self.Exp.transpose(-2, -1), dQ)
    dW_K = torch.bmm(self.Exp.transpose(-2, -1), dK)
    dW_V = torch.bmm(self.Exp.transpose(-2, -1), dV)

    # sum gradients across all heads to get dX
    dX = (torch.bmm(dQ, self.Q_M.transpose(-2, -1)) +
          torch.bmm(dK, self.K_M.transpose(-2, -1)) +
          torch.bmm(dV, self.V_M.transpose(-2, -1))).sum(dim=0)

    return dW_Q, dW_K, dW_V, dW_O, dX


class Layer(nn.Module):
  def __init__(self, input, prev):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(prev, input))
    self.bias = nn.Parameter(torch.randn(input))
    self.activations = None
    self.z = None
    self.inputs = input
    self.prev = prev


class MLP(nn.Module):
  def __init__(self, num_layers:int, layers:list):
    super().__init__()
    self.num = num_layers
    self.layers = nn.ModuleList()
    self.l = layers
    self.input = None
    for i in range(1, self.num + 1):
      self.layers.append(Layer(layers[i], layers[i - 1]))

  def forward_pass(self, input):
    self.input = input
    current = input                          # don't use layers[0] as input holder
    for i in range(0, self.num - 1):         # hidden layers with relu
        self.layers[i].z = (current @ self.layers[i].weights) + self.layers[i].bias
        self.layers[i].activations = relu(self.layers[i].z)
        current = self.layers[i].activations

    last = self.num - 1                      # final layer, no activation
    self.layers[last].z = current @ self.layers[last].weights + self.layers[last].bias
    self.layers[last].activations = self.layers[last].z

    return self.layers[last].activations

  def backpropagation(self, i_g):
    dW = deque()
    db = deque()
    delta = i_g

    for i in range(self.num - 1, -1, -1):
        if i != self.num - 1:
            delta = delta @ self.layers[i+1].weights.T * relu_derivative(self.layers[i].z)
        db.appendleft(delta.sum(dim=0))
        prev_act = self.input if i == 0 else self.layers[i-1].activations
        dW.appendleft(prev_act.T @ delta)

    d_input = delta @ self.layers[0].weights.T
    return dW, db, d_input


class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.input = None
        self.mu = None
        self.var = None
        self.x_hat = None

    def forward_pass(self, E):
        self.input = E
        self.mu = E.sum(dim=-1, keepdim=True) / self.d_model
        self.var = ((E - self.mu) ** 2).sum(dim=-1, keepdim=True) / self.d_model
        self.x_hat = (E - self.mu) / (self.var + 1e-5) ** 0.5
        return self.gamma * self.x_hat + self.beta

    def backpropagation(self, i_g):
        sigma = (self.var + 1e-5) ** 0.5

        d_gamma = (i_g * self.x_hat).sum(dim=0)
        d_beta = i_g.sum(dim=0)

        dx_hat = i_g * self.gamma

        # quotient rule accounting for mu and sigma both depending on X
        dX = (self.gamma / sigma) * (
            dx_hat
            - dx_hat.sum(dim=-1, keepdim=True) / self.d_model
            - self.x_hat * (dx_hat * self.x_hat).sum(dim=-1, keepdim=True) / self.d_model
        )

        return d_gamma, d_beta, dX


class Transformer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = Attention_Block(d_model, num_heads).to(device)
        self.mlp = MLP(3, [d_model, 4*d_model, 4*d_model, d_model]).to(device)
        self.norm1 = LayerNorm(d_model).to(device)
        self.norm2 = LayerNorm(d_model).to(device)
        self.input = None
        self.norm1_out = None
        self.attn_out = None
        self.norm2_out = None
        self.mlp_out = None

    def forward_pass(self, E, mask=False):
        self.input = E

        self.norm1_out = self.norm1.forward_pass(E)
        self.attn_out = self.attention.forward_pass(self.norm1_out, mask)
        E = E + self.attn_out

        self.norm2_out = self.norm2.forward_pass(E)
        self.mlp_out = self.mlp.forward_pass(self.norm2_out)
        E = E + self.mlp_out

        return E

    def backpropagation(self, i_g):
        # residual: gradient copies to both branches, then sums when they rejoin
        d_mlp_out = i_g
        d_E_mid = i_g

        dW_mlp, db_mlp, d_norm2_out = self.mlp.backpropagation(d_mlp_out)
        d_gamma2, d_beta2, d_norm2_in = self.norm2.backpropagation(d_norm2_out)

        d_E_mid = d_E_mid + d_norm2_in

        d_attn_out = d_E_mid
        d_input = d_E_mid

        dW_Q, dW_K, dW_V, dW_O, d_norm1_out = self.attention.backpropagation(d_attn_out)
        d_gamma1, d_beta1, d_norm1_in = self.norm1.backpropagation(d_norm1_out)

        d_input = d_input + d_norm1_in

        grads = {
            'dW_Q': dW_Q, 'dW_K': dW_K, 'dW_V': dW_V, 'dW_O': dW_O,
            'dW_mlp': dW_mlp, 'db_mlp': db_mlp,
            'd_gamma1': d_gamma1, 'd_beta1': d_beta1,
            'd_gamma2': d_gamma2, 'd_beta2': d_beta2,
        }

        return grads, d_input


class LLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, max_seq_len):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, d_model))
        self.pos_encoding = positional_encoding(max_seq_len, d_model).to(device)
        self.blocks = nn.ModuleList([
            Transformer(d_model, num_heads)
            for _ in range(num_blocks)
        ])
        self.unembedding = nn.Parameter(torch.randn(d_model, vocab_size))
        self.token_ids = None
        self.E = None
        self.logits = None
        self.probs = None

    def forward_pass(self, token_ids, mask=False):
      self.token_ids = token_ids
      self.E = self.embedding[token_ids] + self.pos_encoding[:len(token_ids)]
      for block in self.blocks:
          self.E = block.forward_pass(self.E, mask)
      self.logits = self.E @ self.unembedding        # (n, vocab_size)
      self.probs = torch.softmax(self.logits, dim=-1)
      return self.probs

    def evaluate(self, token_ids):
      # same as forward_pass but no mask and returns only the last token's probs
      self.token_ids = token_ids
      self.E = self.embedding[token_ids] + self.pos_encoding[:len(token_ids)]
      for block in self.blocks:
          self.E = block.forward_pass(self.E, mask=False)
      logits = self.E[-1] @ self.unembedding
      probs = torch.softmax(logits, dim=-1)
      return probs

    def loss(self, target_ids):
      # probs[i] predicts token at position i+1
      # so probs[:-1] aligns with target_ids[:-1]
      target_ids = target_ids[:-1]
      n = len(target_ids)
      p_correct = self.probs[:-1][torch.arange(n, device=self.probs.device), target_ids]
      return -torch.log(p_correct + 1e-9).mean()

    def backpropagation(self, target_ids):
      all_grads = []
      target_ids = target_ids[:-1]   # align with probs[:-1]
      n = len(target_ids)
      dev = self.probs.device

      d_logits = torch.zeros_like(self.probs)
      d_logits[:-1] = self.probs[:-1].clone()
      d_logits[:-1][torch.arange(n, device=dev), target_ids] -= 1.0
      d_logits[:-1] /= n

      dW_unembed = self.E.T @ d_logits
      d_E = d_logits @ self.unembedding.T

      for block in reversed(self.blocks):
          grads, d_E = block.backpropagation(d_E)
          all_grads.append(grads)

      d_embedding = torch.zeros_like(self.embedding)
      for t, token_id in enumerate(self.token_ids):
          d_embedding[token_id] += d_E[t]

      return all_grads, dW_unembed, d_embedding


class AdamW:
    def __init__(self, params, lr=3e-4, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0

        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def flatten_grads(self, all_grads, dW_unembed, d_embedding):
        # all_grads is a list of per-block grad dicts in reversed order
        # (block N-1 first, block 0 last) — we reverse it to get forward order
        flat = []

        # embedding and unembedding come first in named_parameters()
        flat.append(d_embedding)   # embedding
        flat.append(dW_unembed)    # unembedding

        # now blocks in forward order
        for block_grads in reversed(all_grads):
            # attention weights — same order as named_parameters()
            flat.append(block_grads['dW_Q'])   # Q_M
            flat.append(block_grads['dW_K'])   # K_M
            flat.append(block_grads['dW_V'])   # V_M
            flat.append(block_grads['dW_O'])   # W_O

            # mlp weights and biases — layers 0, 1, 2 in order
            dW_mlp = list(block_grads['dW_mlp'])   # deque → list
            db_mlp = list(block_grads['db_mlp'])   # deque → list
            for dW, db in zip(dW_mlp, db_mlp):
                flat.append(dW)
                flat.append(db)

            # norm1 then norm2
            flat.append(block_grads['d_gamma1'])   # norm1.gamma
            flat.append(block_grads['d_beta1'])    # norm1.beta
            flat.append(block_grads['d_gamma2'])   # norm2.gamma
            flat.append(block_grads['d_beta2'])    # norm2.beta

        return flat

    def step(self, all_grads, dW_unembed, d_embedding):
        self.t += 1
        grads = self.flatten_grads(all_grads, dW_unembed, d_embedding)

        bc1 = 1 - self.beta1 ** self.t
        bc2 = 1 - self.beta2 ** self.t

        for i, (p, g) in enumerate(zip(self.params, grads)):
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2

            m_hat = self.m[i] / bc1
            v_hat = self.v[i] / bc2

            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)
            p.data -= self.lr * self.weight_decay * p.data


class Tokenizer:
    def __init__(self, text):
        # build vocab from whatever text you give it
        self.chars = sorted(set(text))
        self.vocab_size = len(self.chars)
        self.c2i = {c: i for i, c in enumerate(self.chars)}
        self.i2c = {i: c for i, c in enumerate(self.chars)}

    def encode(self, text):
        return [self.c2i[c] for c in text if c in self.c2i]

    def decode(self, ids):
        return ''.join(self.i2c[i] for i in ids)


class AIAgent:
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, max_seq_len):
        self.max_seq_len = max_seq_len
        self.tokenizer = None  # set later via load_tokenizer()
        self.llm = LLM(vocab_size, d_model, num_heads, num_blocks, max_seq_len).to(device)
        self.optimizer = AdamW(self.llm.parameters(), lr=3e-4, weight_decay=0.01)

    def load_tokenizer(self, text):
        # build tokenizer from any string corpus you pass in
        self.tokenizer = Tokenizer(text)

    def _prepare_input(self, text):
        # encode and clip to max_seq_len
        token_ids = self.tokenizer.encode(text)
        token_ids = token_ids[-self.max_seq_len:]  # keep last max_seq_len tokens
        return torch.tensor(token_ids, dtype=torch.long).to(device)

    def generate(self, prompt, num_tokens=100, temperature=1.0):
        assert self.tokenizer is not None, "Call load_tokenizer(text) first"
        self.llm.eval()

        token_ids = self.tokenizer.encode(prompt)
        token_ids = token_ids[-self.max_seq_len:]

        generated = list(token_ids)

        with torch.no_grad():
            for _ in range(num_tokens):
                input_tensor = torch.tensor(generated[-self.max_seq_len:], dtype=torch.long).to(device)
                probs = self.llm.evaluate(input_tensor)

                # temperature scales confidence — lower = more confident, higher = more random
                probs = probs ** (1.0 / temperature)
                probs = probs / probs.sum()

                next_token = torch.multinomial(probs, num_samples=1).item()
                generated.append(next_token)

        output = self.tokenizer.decode(generated)
        return output

    def train_step(self, text):
        assert self.tokenizer is not None, "Call load_tokenizer(text) first"
        self.llm.train()

        token_ids = self.tokenizer.encode(text)
        if len(token_ids) < 2:
            return None  # need at least 2 tokens to have an input and a target

        # clip to max_seq_len
        token_ids = token_ids[:self.max_seq_len + 1]
        inputs  = token_ids[:-1]   # everything except last token
        targets = token_ids[1:]    # everything except first token

        input_tensor  = torch.tensor(inputs,  dtype=torch.long).to(device)
        target_tensor = torch.tensor(targets, dtype=torch.long).to(device)

        self.llm.forward_pass(input_tensor, mask=True)
        loss = self.llm.loss(target_tensor)
        all_grads, dW_unembed, d_embedding = self.llm.backpropagation(target_tensor)
        self.optimizer.step(all_grads, dW_unembed, d_embedding)

        return loss.item()

    def save(self, path):
        torch.save({
            'model': self.llm.state_dict(),
            'optimizer_m': self.optimizer.m,
            'optimizer_v': self.optimizer.v,
            'optimizer_t': self.optimizer.t,
        }, path)
        print(f"Saved to {path}")

    def load(self, path):
        checkpoint = torch.load(path, map_location=device)
        self.llm.load_state_dict(checkpoint['model'])
        self.optimizer.m = checkpoint['optimizer_m']
        self.optimizer.v = checkpoint['optimizer_v']
        self.optimizer.t = checkpoint['optimizer_t']
        print(f"Loaded from {path}")

print("Everything Works Hopefully")

Everything Works Hopefully


In [6]:
dummy_text = "hello how are you. the wavefunction collapses. energy equals mass times c squared. hi good morning."

temp_tokenizer = Tokenizer(dummy_text)
print(f"vocab size: {temp_tokenizer.vocab_size}")

agent = AIAgent(
    vocab_size  = temp_tokenizer.vocab_size,
    d_model     = 32,
    num_heads   = 2,
    num_blocks  = 2,
    max_seq_len = 64
)
agent.tokenizer = temp_tokenizer

print("\n--- generate test ---")
output = agent.generate("hello", num_tokens=20)
print(f"output: {output}")

print("\n--- train step test ---")
loss = agent.train_step(dummy_text)   # same text as tokenizer was built from
print(f"loss: {loss:.4f}")

vocab size: 23

--- generate test ---
output: hellopfffftpfffffffffffff

--- train step test ---
loss: 20.0654


In [14]:
print(agent.generate("oneplusone", num_tokens=21))

oneplusonefffffffffffitfnfitfca
